# PySpark Data Processing - Assignment 6

This notebook contains my code solutions and technical explanations for the Spark Data Processing Assignment. It demonstrates spark setup, basic schema parsing, data cleaning, filtering under multiple conditions, type casting, file IO performance, and architectural best practices.

---


In [1]:
import os
import sys
from pyspark.sql import SparkSession
import pyspark.sql.functions as func

# Configure python path for the local spark cluster
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = SparkSession.builder \
    .appName("PySpark_Data_Processing_Assignment6") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession is ready! Spark version is:", spark.version)


SparkSession is ready! Spark version is: 4.2.0


## Q1: Roles of Driver, Cluster Manager, and Executor in Spark

The Spark runtime architecture coordinates processing tasks across cluster hardware through three distinct components:

1. **Driver**:
   Runs the `main()` method of your Python script or Jupyter process and initializes the `SparkSession`. It parses code, translates transformations into a Directed Acyclic Graph (DAG) plan, optimizes it, builds execution tasks, and distributes them to executors.
   
2. **Cluster Manager**:
   Mediates resource allocation. It can run Standalone, YARN, or Kubernetes. It monitors workers and launches executor processes upon request by the driver.

3. **Executor**:
   Individual worker JVM processes running on cluster nodes. They execute code tasks assigned to them, store cached partitions in RAM/disk, and report status/outputs back to the driver.


In [2]:
sc = spark.sparkContext
print("Active App ID:", sc.applicationId)
print("Master URL:", sc.master)


Active App ID: local-1785052965143
Master URL: local[*]


## Q2: How Lazy Evaluation Enhances Spark Performance

Lazy evaluation defers execution of transformations (`filter`, `select`, `join`, etc.) until a concrete action is requested. When you write transformation commands, Spark only updates a logical execution plan (DAG lineage).

### Performance Benefits:
- **Optimization Opportunities**: Because Spark views the full query flow, the Catalyst Optimizer reorders queries, pushes filters upstream, and prunes unused columns before executing code.
- **Data Pipelining**: Spark runs multiple narrow transformations in memory as a single stage, avoiding disk I/O between intermediate steps.
- **Resource Savings**: Computing steps only when needed avoids unnecessary resource consumption on records that are eventually filtered out.


In [3]:
raw_df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
query_plan = raw_df.filter(func.col("category") == "Electronics").select("product_id", "price")

print("Lazy execution plan registered:")
query_plan.explain(True)


== Parsed Logical Plan ==
'Project ['product_id, 'price]
+- Filter (category#18 = Electronics)
   +- Relation [product_id#17,category#18,price#19,old_name#20,status#21,amount#22,base_price#23,region#24,priority#25,user_id#26] csv

== Analyzed Logical Plan ==
product_id: string, price: double
Project [product_id#17, price#19]
+- Filter (category#18 = Electronics)
   +- Relation [product_id#17,category#18,price#19,old_name#20,status#21,amount#22,base_price#23,region#24,priority#25,user_id#26] csv

== Optimized Logical Plan ==
Project [product_id#17, price#19]
+- Filter (isnotnull(category#18) AND (category#18 = Electronics))
   +- Relation [product_id#17,category#18,price#19,old_name#20,status#21,amount#22,base_price#23,region#24,priority#25,user_id#26] csv

== Physical Plan ==
*(1) Project [product_id#17, price#19]
+- *(1) Filter (isnotnull(category#18) AND (category#18 = Electronics))
   +- FileScan csv [product_id#17,category#18,price#19] Batched: false, DataFilters: [isnotnull(catego

## Q3: Reading a CSV with Header and inferSchema

### Code Solution
Command to load `"data/source.csv"` with header and schema inference enabled:


In [4]:
sales_df = spark.read.csv("data/source.csv", header=True, inferSchema=True)
sales_df.printSchema()
sales_df.show(5)


root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- old_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------+---------------+------+--------------+---------+------+----------+------+--------+-------+
|product_id|       category| price|      old_name|   status|amount|base_price|region|priority|user_id|
+----------+---------------+------+--------------+---------+------+----------+------+--------+-------+
|      P901|    Electronics|1200.0|  Legacy Alpha|Completed|2400.0|    1000.0| North|    High|   U201|
|      P902|      Furniture|  85.0|   Legacy Beta|Completed| 170.0|      75.0|  East|  Medium|   U202|
|      P903|    Electronics| 450.0|  Legacy Gamma|  Pending| 450.0|     400.0| South|

## Q4: CSV vs. Parquet (Row-Based vs. Columnar Storage)

- **CSV**: A text-based, row-oriented format. Each record is written in a new line. To read a single column, Spark must load and read all columns and rows from disk, which causes heavy I/O overhead. CSVs have no native compression or index metadata.
- **Parquet**: A binary, columnar storage format. Data is stored by columns instead of rows. It includes footer metadata (statistics on min, max, null counts per block) and native dictionary encoding.

### Performance Implications:
1. **Column Skip**: Parquet reads only the specific byte segments representing the columns in a select statement, saving massive read I/O.
2. **Superior Compression**: Column data is of homogeneous types, enabling high compression ratios using Snappy or Gzip.
3. **Optimized Filtering**: Block statistics allow Spark to bypass whole row groups entirely when filter predicates don't overlap with column boundaries.


## Q5: Select columns 'product_id' and 'price' where category is 'Electronics'

### Code Solution


In [6]:
electronics_only = sales_df.filter(func.col("category") == "Electronics").select("product_id", "price")
electronics_only.show()


+----------+------+
|product_id| price|
+----------+------+
|      P901|1200.0|
|      P903| 450.0|
|      P906| 99.99|
|      P908| 150.0|
+----------+------+


## Q6: Rename Column and Cast Type

### Code Solution
Renaming `old_name` to `new_name` and casting the `price` column from String to Double:


In [7]:
revised_df = sales_df.withColumnRenamed("old_name", "new_name") \
    .withColumn("price", func.col("price").cast("double"))

revised_df.printSchema()
revised_df.select("product_id", "new_name", "price").show(5)


root
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)
 |-- new_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- base_price: double (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)
 |-- user_id: string (nullable = true)

+----------+--------------+------+
|product_id|      new_name| price|
+----------+--------------+------+
|      P901|  Legacy Alpha|1200.0|
|      P902|   Legacy Beta|  85.0|
|      P903|  Legacy Gamma| 450.0|
|      P904|  Legacy Delta|  35.5|
|      P905|Legacy Epsilon| 12.99|
+----------+--------------+------+
only showing top 5 rows


## Q7: Fault Tolerance via Lineage Graph (DAG)

Spark achieves fault tolerance by tracking dataset transformations through a logical Directed Acyclic Graph (DAG) lineage. Because DataFrames are immutable, Spark does not need to replicate data blocks across nodes.

If a worker node crashes:
1. Spark knows which partition was stored on that node and lost.
2. It looks up the DAG lineage graph for that specific partition.
3. Spark schedules tasks to recompute *only* the lost partition from the nearest parent dataset or source file.
4. Other partitions on active nodes are not affected, preventing the need to restart the entire application execution.


## Q8: Filter orders where status is 'Completed' AND amount is greater than 1000

### Code Solution


In [8]:
completed_high_sales = sales_df.filter((func.col("status") == "Completed") & (func.col("amount") > 1000))
completed_high_sales.show()


+----------+-----------+------+------------+---------+------+----------+------+--------+-------+
|product_id|   category| price|    old_name|   status|amount|base_price|region|priority|user_id|
+----------+-----------+------+------------+---------+------+----------+------+--------+-------+
|      P901|Electronics|1200.0|Legacy Alpha|Completed|2400.0|    1000.0| North|    High|   U201|
|      P907|  Furniture| 300.0|  Legacy Eta|Completed|1500.0|     250.0|  West|  Medium|   U207|
+----------+-----------+------+------------+---------+------+----------+------+--------+-------+


## Q9: Predicate Pushdown in Parquet

**Predicate Pushdown** refers to the optimization where filter operations are applied at the storage tier before loading records into Spark Executor JVM memory.

Parquet files organize data into Row Groups and store min/max value boundaries inside file footers. When Spark executes a query with a filter (e.g. `amount > 1000`), the Parquet reader scans the file footers first. If a Row Group's metadata shows a maximum `amount` of `500`, Spark skips reading the row group. Only relevant row groups enter Spark memory, lowering network and RAM overhead.


## Q10: Add New Column with Tax (18% tax)

### Code Solution
Creating a new column `final_price` which calculates `base_price * 1.18`:


In [10]:
tax_added_df = sales_df.withColumn("final_price", func.round(func.col("base_price") * 1.18, 2))
tax_added_df.select("product_id", "base_price", "final_price").show(5)


+----------+----------+-----------+
|product_id|base_price|final_price|
+----------+----------+-----------+
|      P901|    1000.0|     1180.0|
|      P902|      75.0|       88.5|
|      P903|     400.0|      472.0|
|      P904|      30.0|       35.4|
|      P905|      10.0|       11.8|
+----------+----------+-----------+
only showing top 5 rows


## Q11: Transformations vs. Actions in Spark

- **Transformations**: Operations that take an existing DataFrame and return a new DataFrame. They do not trigger computations (lazy execution).
  - *Examples*: `.filter()`, `.select()`, `.groupBy()`, `.join()`
- **Actions**: Operations that force Spark to compile and execute the DAG lineage, returning values to the driver or exporting to disk.
  - *Examples*: `.count()`, `.collect()`, `.show()`, `.write`


## Q12: Data Pipeline (Load Parquet -> Clean Nulls -> Write CSV)

### Code Solution
Command to load a Parquet file, filter out null `user_id` values, and output to a CSV:


In [11]:
# PySpark syntax requested:
# (spark.read.parquet("path/to/input")
#      .filter(func.col("user_id").isNotNull())
#      .write.mode("overwrite")
#      .option("header", "true")
#      .csv("path/to/output"))

# Executing pipeline on local parquet:
clean_users_df = spark.read.parquet("data/input_parquet/data.parquet")
clean_users_df = clean_users_df.filter(func.col("user_id").isNotNull())

clean_users_df.show()
clean_users_df.toPandas().to_csv("output/filtered_users_output.csv", index=False)
print("Pipeline run successful. Output saved.")


+----------+---------------+------+--------------+---------+------+----------+------+--------+-------+
|product_id|       category| price|      old_name|   status|amount|base_price|region|priority|user_id|
+----------+---------------+------+--------------+---------+------+----------+------+--------+-------+
|      P901|    Electronics|1200.0|  Legacy Alpha|Completed|2400.0|    1000.0| North|    High|   U201|
|      P902|      Furniture|  85.0|   Legacy Beta|Completed| 170.0|      75.0|  East|  Medium|   U202|
|      P903|    Electronics| 450.0|  Legacy Gamma|  Pending| 450.0|     400.0| South|    High|   U203|
|      P905|Office Supplies| 12.99|Legacy Epsilon|Completed| 12.99|      10.0|  West|     Low|   U205|
|      P907|      Furniture| 300.0|    Legacy Eta|Completed|1500.0|     250.0|  West|  Medium|   U207|
|      P908|    Electronics| 150.0|  Legacy Theta|  Pending| 300.0|     130.0|  East|     Low|   U208|
|      P909|       Clothing|  95.0|   Legacy Iota|Completed| 190.0|      

## Q13: Client Mode vs. Cluster Mode in Spark Deployment

- **Client Mode**:
  - The Driver process runs directly on the machine where the job was launched (e.g., your local machine, notebook server).
  - Executors run on worker nodes.
  - *Use Case*: Good for interactive development, repl, and notebooks because you see stdout instantly.
- **Cluster Mode**:
  - The Driver process is executed inside an Application Master container on a worker node within the cluster.
  - The client machine launches the job and exits.
  - *Use Case*: Production pipelines where you don't want the job to crash if your local computer disconnects.


## Q14: Multi-Condition Filtering (OR Logic)

### Code Solution
Select rows where the `region` is `'North'` OR the `priority` is `'High'`:


In [13]:
north_or_high_priority = sales_df.filter((func.col("region") == "North") | (func.col("priority") == "High"))
north_or_high_priority.show()


+----------+-----------+------+------------+---------+------+----------+------+--------+-------+
|product_id|   category| price|    old_name|   status|amount|base_price|region|priority|user_id|
+----------+-----------+------+------------+---------+------+----------+------+--------+-------+
|      P901|Electronics|1200.0|Legacy Alpha|Completed|2400.0|    1000.0| North|    High|   U201|
|      P903|Electronics| 450.0|Legacy Gamma|  Pending| 450.0|     400.0| South|    High|   U203|
|      P904|   Clothing|  35.5|Legacy Delta|Cancelled|  71.0|      30.0| North|     Low|   NULL|
|      P906|Electronics| 99.99| Legacy Zeta|Completed|199.98|      80.0| North|    High|   NULL|
|      P909|   Clothing|  95.0| Legacy Iota|Completed| 190.0|      80.0| North|    High|   U209|
+----------+-----------+------+------------+---------+------+----------+------+--------+-------+


## Q15: Why `.show(5)` is Safer Than `.collect()`

- **`.collect()`**: Pulls all records from every partition across the network and holds them in memory as a single local Python list on the Driver node. On multi-terabyte datasets, this causes an OutOfMemory (OOM) error, crashing the Driver JVM.
- **`.show(5)`**: Fetches only the first 5 records from the first active partition. It uses minor CPU and network bandwidth, keeping driver memory usage safe during interactive data exploration.


## Superstore Performance Pipeline

Below is a data processing workflow on the Superstore dataset, showing aggregate metrics of Sales by Category and Sub-Category.


In [14]:
superstore_df = spark.read.csv("data/Sample - Superstore.csv", header=True, inferSchema=True, escape='"')

print("Superstore Schema:")
superstore_df.printSchema()

print("\nSales Summary by Category and Sub-Category:")
metrics_df = superstore_df.groupBy("Category", "Sub-Category") \
    .agg(
        func.round(func.sum("Sales"), 2).alias("Total_Revenue"),
        func.round(func.avg("Sales"), 2).alias("Average_Sale"),
        func.count("Order ID").alias("Number_Of_Orders")
    ).orderBy(func.desc("Total_Revenue"))

metrics_df.show(10, truncate=False)
metrics_df.toPandas().to_csv("output/superstore_analytics.csv", index=False)
print("Saved summary metrics to output/superstore_analytics.csv")


root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)


Sales Summary by Category and Sub-Category:
+---------------+------------+-------------+------------+----------------+
|Category       |Sub-Cat